# Internship Task: User Authentication & Chatbot API

Since the intern already knows Python and FastAPI but is new to PostgreSQL, this project is divided into small milestones. Each task builds on the previous one.

## Objective
Build a FastAPI backend that provides:
- User Registration (Sign Up)
- User Login
- JWT Authentication
- Chatbot API
- Store user conversations in PostgreSQL


## Phase 1 — PostgreSQL Setup

**Task 1**

Install PostgreSQL.

**Learn:**
- What is a database?
- What is a table?
- What is a row?
- What is a primary key?
- What is a foreign key?

**Deliverable:**
- PostgreSQL installed
- Create a database named: `chatbot_db`


In [ ]:
CREATE DATABASE chatbot_db;

## Phase 2 — Connect FastAPI with PostgreSQL

**Task 2**

Install required packages:
- fastapi
- uvicorn
- sqlalchemy
- psycopg2-binary
- alembic
- python-jose
- passlib[bcrypt]
- python-multipart

Create `database.py`.

**Requirements:**
- Connect FastAPI to PostgreSQL
- Test database connection successfully

**Deliverable:** Database connected successfully.


In [ ]:
# Install packages (run in terminal, not notebook, for a real project)
# pip install fastapi uvicorn sqlalchemy psycopg2-binary alembic python-jose passlib[bcrypt] python-multipart


In [ ]:
# database.py

from sqlalchemy import create_engine
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker

SQLALCHEMY_DATABASE_URL = "postgresql://<username>:<password>@localhost/chatbot_db"

engine = create_engine(SQLALCHEMY_DATABASE_URL)
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)

Base = declarative_base()

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()


## Phase 3 — Design Database

Create the following tables.

### Users Table
| Column | Type |
|---|---|
| id | Integer |
| full_name | String |
| email | String (Unique) |
| password | Hashed Password |
| created_at | Timestamp |

### Chat History Table
| Column | Type |
|---|---|
| id | Integer |
| user_id | Foreign Key |
| user_message | Text |
| bot_response | Text |
| created_at | Timestamp |

**Relationship:** One User → Many Chat Messages


## Phase 4 — SQLAlchemy Models

Create models:
- `User`
- `ChatHistory`

**Requirements:**
- Define relationships
- Create tables using SQLAlchemy

**Deliverable:** Tables visible inside PostgreSQL.


In [ ]:
# models.py

from sqlalchemy import Column, Integer, String, Text, ForeignKey, DateTime
from sqlalchemy.orm import relationship
from sqlalchemy.sql import func
from database import Base

class User(Base):
    __tablename__ = "users"

    id = Column(Integer, primary_key=True, index=True)
    full_name = Column(String, nullable=False)
    email = Column(String, unique=True, index=True, nullable=False)
    password = Column(String, nullable=False)
    created_at = Column(DateTime(timezone=True), server_default=func.now())

    chats = relationship("ChatHistory", back_populates="user")


class ChatHistory(Base):
    __tablename__ = "chat_history"

    id = Column(Integer, primary_key=True, index=True)
    user_id = Column(Integer, ForeignKey("users.id"))
    user_message = Column(Text, nullable=False)
    bot_response = Column(Text, nullable=False)
    created_at = Column(DateTime(timezone=True), server_default=func.now())

    user = relationship("User", back_populates="chats")


In [ ]:
# Create tables (e.g. in main.py or a setup script)

# from database import engine, Base
# import models
# Base.metadata.create_all(bind=engine)


## Phase 5 — User Registration API

**Endpoint:** `POST /signup`

**Input**
```json
{
    "full_name":"John",
    "email":"john@gmail.com",
    "password":"12345678"
}
```

**Requirements**
- Validate email
- Check duplicate email
- Hash password using bcrypt
- Save user into database

**Response**
```json
{
    "message":"User registered successfully"
}
```


In [ ]:
# schemas.py (relevant excerpt)

from pydantic import BaseModel, EmailStr

class UserCreate(BaseModel):
    full_name: str
    email: EmailStr
    password: str


In [ ]:
# utils/hashing.py

from passlib.context import CryptContext

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

def hash_password(password: str) -> str:
    return pwd_context.hash(password)

def verify_password(plain_password: str, hashed_password: str) -> bool:
    return pwd_context.verify(plain_password, hashed_password)


In [ ]:
# routers/auth.py (signup excerpt)

from fastapi import APIRouter, Depends, HTTPException
from sqlalchemy.orm import Session
from database import get_db
import models, schemas
from utils.hashing import hash_password

router = APIRouter()

@router.post("/signup")
def signup(user: schemas.UserCreate, db: Session = Depends(get_db)):
    existing_user = db.query(models.User).filter(models.User.email == user.email).first()
    if existing_user:
        raise HTTPException(status_code=400, detail="Email already registered")

    new_user = models.User(
        full_name=user.full_name,
        email=user.email,
        password=hash_password(user.password),
    )
    db.add(new_user)
    db.commit()
    db.refresh(new_user)

    return {"message": "User registered successfully"}


## Phase 6 — Login API

**Endpoint:** `POST /login`

**Input**
```json
{
    "email":"john@gmail.com",
    "password":"12345678"
}
```

**Requirements**
- Verify email
- Verify password
- Generate JWT token
- Return access token

**Example Response**
```json
{
    "access_token":"...",
    "token_type":"bearer"
}
```


In [ ]:
# utils/token.py

from datetime import datetime, timedelta
from jose import jwt

SECRET_KEY = "CHANGE_THIS_SECRET_KEY"
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 60

def create_access_token(data: dict, expires_delta: timedelta = None):
    to_encode = data.copy()
    expire = datetime.utcnow() + (expires_delta or timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES))
    to_encode.update({"exp": expire})
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)


In [ ]:
# routers/auth.py (login excerpt)

from fastapi import APIRouter, Depends, HTTPException
from sqlalchemy.orm import Session
from database import get_db
import models, schemas
from utils.hashing import verify_password
from utils.token import create_access_token

@router.post("/login")
def login(credentials: schemas.UserLogin, db: Session = Depends(get_db)):
    user = db.query(models.User).filter(models.User.email == credentials.email).first()
    if not user or not verify_password(credentials.password, user.password):
        raise HTTPException(status_code=401, detail="Invalid email or password")

    access_token = create_access_token(data={"sub": str(user.id)})
    return {"access_token": access_token, "token_type": "bearer"}


## Phase 7 — Protected Routes

Create JWT authentication.

**Requirement**
- Only authenticated users can access: `POST /chat`
- If token is invalid, return `401 Unauthorized`


In [ ]:
# dependencies.py

from fastapi import Depends, HTTPException
from fastapi.security import OAuth2PasswordBearer
from jose import jwt, JWTError
from sqlalchemy.orm import Session
from database import get_db
import models
from utils.token import SECRET_KEY, ALGORITHM

oauth2_scheme = OAuth2PasswordBearer(tokenUrl="login")

def get_current_user(token: str = Depends(oauth2_scheme), db: Session = Depends(get_db)):
    credentials_exception = HTTPException(status_code=401, detail="Could not validate credentials")
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        user_id: str = payload.get("sub")
        if user_id is None:
            raise credentials_exception
    except JWTError:
        raise credentials_exception

    user = db.query(models.User).filter(models.User.id == int(user_id)).first()
    if user is None:
        raise credentials_exception
    return user


## Phase 8 — Chatbot API

**Endpoint:** `POST /chat`

**Input**
```json
{
    "message":"Hello"
}
```

For now, create a dummy chatbot (no AI integration yet).

**Example**
- Input: `Hello`
- Output: `Hello! How can I help you today?`


In [ ]:
# services/chatbot.py

def get_bot_response(message: str) -> str:
    message = message.lower().strip()
    if "hello" in message or "hi" in message:
        return "Hello! How can I help you today?"
    elif "bye" in message:
        return "Goodbye! Have a great day."
    else:
        return "I'm still learning. Can you rephrase that?"


## Phase 9 — Store Chat History

Whenever the chatbot is called, save:
- User ID
- User Message
- Bot Response
- Timestamp

into PostgreSQL.

**Example**
| User | Message | Response |
|---|---|---|
| 1 | Hello | Hi! |
| 1 | How are you? | I'm fine. |


In [ ]:
# routers/chat.py (POST /chat excerpt)

from fastapi import APIRouter, Depends
from sqlalchemy.orm import Session
from database import get_db
import models, schemas
from dependencies import get_current_user
from services.chatbot import get_bot_response

router = APIRouter()

@router.post("/chat")
def chat(
    request: schemas.ChatRequest,
    db: Session = Depends(get_db),
    current_user: models.User = Depends(get_current_user),
):
    bot_response = get_bot_response(request.message)

    chat_entry = models.ChatHistory(
        user_id=current_user.id,
        user_message=request.message,
        bot_response=bot_response,
    )
    db.add(chat_entry)
    db.commit()
    db.refresh(chat_entry)

    return {"response": bot_response}


## Phase 10 — Get Chat History

**Endpoint:** `GET /chat/history`

**Requirements**
- Return only the logged-in user's chat history.

**Example Response**
```json
[
    {
        "message":"Hello",
        "response":"Hi!",
        "time":"2026-06-29"
    }
]
```


In [ ]:
# routers/chat.py (GET /chat/history excerpt)

@router.get("/chat/history")
def get_chat_history(
    db: Session = Depends(get_db),
    current_user: models.User = Depends(get_current_user),
):
    history = (
        db.query(models.ChatHistory)
        .filter(models.ChatHistory.user_id == current_user.id)
        .order_by(models.ChatHistory.created_at)
        .all()
    )

    return [
        {
            "message": h.user_message,
            "response": h.bot_response,
            "time": h.created_at,
        }
        for h in history
    ]


## Phase 11 — Project Structure

The project should follow this structure:

```
app/
│
├── main.py
├── database.py
├── models.py
├── schemas.py
├── auth.py
├── dependencies.py
├── routers/
│   ├── auth.py
│   └── chat.py
│
├── services/
│   └── chatbot.py
│
└── utils/
    ├── hashing.py
    └── token.py
```


In [ ]:
# main.py

from fastapi import FastAPI
from database import engine, Base
import models
from routers import auth, chat

Base.metadata.create_all(bind=engine)

app = FastAPI(title="Chatbot Auth API")

app.include_router(auth.router, tags=["Auth"])
app.include_router(chat.router, tags=["Chat"])

@app.get("/")
def root():
    return {"message": "Chatbot Auth API is running"}


## Phase 12 — API Documentation

Verify all endpoints using FastAPI Swagger UI (`/docs`).

**Required endpoints:**
- `POST /signup`
- `POST /login`
- `POST /chat`
- `GET /chat/history`


In [ ]:
# Run the server (in terminal, not notebook):
# uvicorn main:app --reload
# Then open: http://127.0.0.1:8000/docs


## Bonus Tasks (Optional)

If all required tasks are completed successfully, implement one or more of the following:
- Add input validation using Pydantic (minimum password length, valid email format).
- Add pagination to the chat history endpoint.
- Create a `DELETE /chat/history` endpoint to clear the logged-in user's chat history.
- Add logging for API requests and errors.
- Write unit tests for the authentication and chat endpoints using pytest.


## Evaluation Criteria

| Criteria | Weight |
|---|---|
| Project structure and code organization | 20% |
| PostgreSQL integration | 20% |
| Sign-up and login implementation | 20% |
| JWT authentication | 15% |
| Chat history storage and retrieval | 15% |
| Code quality, validation, and error handling | 10% |


## Learning Outcomes

By completing this assignment, the intern should be able to:
- Understand PostgreSQL fundamentals.
- Connect FastAPI to a relational database using SQLAlchemy.
- Design and implement database tables and relationships.
- Build secure authentication using hashed passwords and JWT.
- Create protected REST APIs.
- Store and retrieve user-specific data from PostgreSQL.
- Organize a FastAPI project using a clean, modular architecture.
